<a href="https://colab.research.google.com/github/fanwenlin/TDDC17-lab/blob/main/lab6/planning_lab_stub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Planning Lab

In [1]:
!pip install unified-planning==1.1.0 up_fast_downward==0.4.1 matplotlib==3.7.1  # for visualisation in this notebook

In [2]:
!pip install matplotlib==3.7.1  # for visualisation in this notebook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 14.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for matplotlib: filename=matplotlib-3.7.1-cp312-cp312-linux_x86_64.whl size=11092757 sha256=d9f4bd41a2bbb46ac9913c22a0c436ad6e37aadbd375b21bdf4acbee96038b52
  Stored in directory: /root/.cache/pip/wheels/1c/06/fa/3453aac11411fac092c1bdfe52815f2f6969a42700d977e62f
Successfully built matplotlib
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
arviz 0.22.0 requires matplotlib>=3.8, but you have matplotlib 3.7.1 which is incompatible.
plotnine 0.14.5 requires matplotlib>

In [2]:
from unified_planning.shortcuts import *

import unified_planning as up

up.shortcuts.get_environment().credits_stream = None

## Part (a): Model the Task

The code below models a simpler version of the Household Robot domain with two rooms and one open door between them. The robot is in room A initially and needs to move to room B. You can use it as a starting point for your own solution to the full Household Robot task.

In [23]:
import unified_planning as up
from unified_planning.shortcuts import *
from unified_planning.model.metrics import MinimizeActionCosts


def get_planning_task():
    Room, Door, Key = UserType("Room"), UserType("Door"), UserType("Key")
    bot_at = Fluent("bot_at", BoolType(), r=Room)  # bot's location
    door_between = Fluent(
        "door_between", BoolType(), r1=Room, d=Door, r2=Room
    )  # door between rooms, constant actually
    unlocked = Fluent(
        "unlocked", BoolType(), d=Door
    )  # if the door is unlocked with key
    key_at = Fluent("key_at", BoolType(), k=Key, r=Room)  # key's location
    have = Fluent("have", BoolType(), k=Key)  # if the bot holds the key
    fits = Fluent("fits", BoolType(), k=Key, d=Door)  # if the key can open the door

    P = Problem("house_keys")

    # define rooms
    KITCHEN, LIVING, CORRIDOR, BATHROOM, LOBBY, OUT = [
        Object(n, Room) for n in ["Kitchen", "Living", "Corridor", "Bathroom", "Lobby", "Out"]
    ]

    # define doors
    dK, dL, dB, dC, dF = [Object(n, Door) for n in ["dK", "dL", "dB", "dC", "dF"]]
    # define keys
    kK, kL, kB, kC, kF, kall = [
        Object(n, Key) for n in ["kK", "kL", "kB", "kC", "kF", "kALL"]
    ]
    P.add_objects(
        [
            KITCHEN,
            LIVING,
            CORRIDOR,
            BATHROOM,
            LOBBY,
            OUT,
            dK,
            dL,
            dB,
            dC,
            dF,
            kK,
            kL,
            kB,
            kC,
            kF,
            kall,
        ]
    )

    P.add_fluent(bot_at, default_initial_value=False)
    P.add_fluent(door_between, default_initial_value=False)
    P.add_fluent(unlocked, default_initial_value=False)
    P.add_fluent(key_at, default_initial_value=False)
    P.add_fluent(have, default_initial_value=False)
    P.add_fluent(fits, default_initial_value=False)

    # doors' location
    conns = [
        (KITCHEN, dK, LIVING),
        (LIVING, dL, CORRIDOR),
        (CORRIDOR, dB, BATHROOM),
        (CORRIDOR, dC, LOBBY),
        (LOBBY, dF, OUT),
    ]
    for r1, d, r2 in conns:
        P.set_initial_value(door_between(r1, d, r2), True)
        P.set_initial_value(door_between(r2, d, r1), True)

    P.set_initial_value(bot_at(LIVING), True)  # Start Point: Living Room

    # set key's location
    for k, d in [(kK, dK), (kL, dL), (kB, dB), (kC, dC), (kF, dF)]:
        P.set_initial_value(fits(k, d), True)

    # which door the key can open
    # kall can open all the doors
    # others have their corresponding door
    for d in [dK, dL, dB, dC, dF]:
        P.set_initial_value(fits(kall, d), True)

    P.set_initial_value(key_at(kK, LIVING), True)
    P.set_initial_value(key_at(kL, LIVING), True)
    P.set_initial_value(key_at(kC, BATHROOM), True)
    P.set_initial_value(key_at(kF, BATHROOM), True)
    P.set_initial_value(key_at(kall, KITCHEN), True)

    # actions
    ## move from room r1 to room r2
    move = InstantaneousAction("move", r1=Room, d=Door, r2=Room)
    r1, d, r2 = move.parameter("r1"), move.parameter("d"), move.parameter("r2")
    move.add_precondition(bot_at(r1))
    move.add_precondition(door_between(r1, d, r2))
    move.add_precondition(unlocked(d))
    move.add_effect(bot_at(r1), False)
    move.add_effect(bot_at(r2), True)
    P.add_action(move)

    # pick up key k in this room r
    pickup = InstantaneousAction("pickup", k=Key, r=Room)
    k, r = pickup.parameter("k"), pickup.parameter("r")
    pickup.add_precondition(bot_at(r))
    pickup.add_precondition(key_at(k, r))
    pickup.add_effect(key_at(k, r), False)
    pickup.add_effect(have(k), True)
    P.add_action(pickup)

    # unlock the door between r1 and r2, with key k, from r
    unlock = InstantaneousAction("unlock", d=Door, r=Room, k=Key, r2=Room)
    d, r, k, r2 = (
        unlock.parameter("d"),
        unlock.parameter("r"),
        unlock.parameter("k"),
        unlock.parameter("r2"),
    )
    unlock.add_precondition(bot_at(r))
    unlock.add_precondition(door_between(r, d, r2))
    unlock.add_precondition(Or(have(k), Bool(False)))
    unlock.add_precondition(fits(k, d))
    unlock.add_effect(unlocked(d), True)
    P.add_action(unlock)

    P.add_goal(unlocked(dF))
    # P.add_goal(bot_at(LOBBY))

    P.add_quality_metric(
        MinimizeActionCosts(
            {move: Int(1), pickup: Int(1), unlock: Int(1)}, default=Int(1)
        )
    )
    return P


problem = get_planning_task()


## Part (b): Find a (possibly suboptimal) plan

Solve the task with greedy best-first search using the FF heuristic. The example code below uses the h^add heuristic. You need to inspect the output to stdout to see the heuristic value of the initial state.

In [24]:
params = {
    "fast_downward_search_config": "eager_greedy([add()])"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem)
    if result.status == up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING:
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found.")
        print(f'Result status: {result.status}')
        print(result.log_messages)

Found a plan of length: 11
SequentialPlan:
    pickup(kK, Living)
    unlock(dK, Living, kK, Kitchen)
    pickup(kL, Living)
    unlock(dL, Living, kL, Corridor)
    move(Living, dK, Kitchen)
    pickup(kALL, Kitchen)
    move(Kitchen, dK, Living)
    move(Living, dL, Corridor)
    unlock(dC, Corridor, kALL, Lobby)
    move(Corridor, dC, Lobby)
    unlock(dF, Lobby, kALL, Out)
Plan cost: {minimize actions-cost: {'move': 1, 'pickup': 1, 'unlock': 1, 'default': 1}: 11}


In [25]:
# change params to FF
params = {
    "fast_downward_search_config": "eager_greedy([ff()])"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem)
    if result.status == up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING:
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found.")
        print(f'Result status: {result.status}')
        print(result.log_messages)

Found a plan of length: 11
SequentialPlan:
    pickup(kK, Living)
    unlock(dK, Living, kK, Kitchen)
    pickup(kL, Living)
    unlock(dL, Living, kL, Corridor)
    move(Living, dK, Kitchen)
    pickup(kALL, Kitchen)
    move(Kitchen, dK, Living)
    move(Living, dL, Corridor)
    unlock(dC, Corridor, kALL, Lobby)
    move(Corridor, dC, Lobby)
    unlock(dF, Lobby, kALL, Out)
Plan cost: {minimize actions-cost: {'move': 1, 'pickup': 1, 'unlock': 1, 'default': 1}: 11}


## Part (c): Find an optimal plan

Solve the task with A* using the `iPDB` heuristic.

In [27]:
# change params to A* with iPDB heuristic
params = {
    "fast_downward_search_config": "astar(ipdb())"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem)
    if result.status == up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING:
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found.")
        print(f'Result status: {result.status}')
        print(result.log_messages)

Found a plan of length: 10
SequentialPlan:
    pickup(kK, Living)
    unlock(dK, Living, kK, Kitchen)
    move(Living, dK, Kitchen)
    pickup(kALL, Kitchen)
    move(Kitchen, dK, Living)
    unlock(dL, Living, kALL, Corridor)
    move(Living, dL, Corridor)
    unlock(dC, Corridor, kALL, Lobby)
    move(Corridor, dC, Lobby)
    unlock(dF, Lobby, kALL, Out)
Plan cost: {minimize actions-cost: {'move': 1, 'pickup': 1, 'unlock': 1, 'default': 1}: 10}
